In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import CamembertModel, AutoTokenizer, CLIPVisionModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

# =====================
# CONFIG
# =====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

BATCH_SIZE = 16
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
ACCUM_STEPS = 2

BACKBONE_LR = 2e-6
CLASSIFIER_LR = 5e-5

MAX_TEXT_LEN = 128

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"

# =====================
# PATHS
# =====================
BASE_DIR = Path.cwd().parent.parent.parent.parent
DATA_DIR = BASE_DIR / "data"
IMAGE_DIR = DATA_DIR / "images" / "image_train"

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# =====================
# LOAD DATA
# =====================
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
train_df = pd.read_csv(SPLIT_DIR / "train_split.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

# =====================
# TOKENIZER / PROCESSOR
# =====================
tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
processor = CLIPProcessor.from_pretrained(CLIP_ID)

# =====================
# DATASET
# =====================
class RakutenDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = IMAGE_DIR / f"image_{row['imageid']}_product_{row['productid']}.jpg"
        image = Image.open(img_path).convert("RGB")

        text = f"{row.get('designation','')} {row.get('description','')}"

        txt = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_TEXT_LEN,
            return_tensors="pt"
        )

        img = processor(images=image, return_tensors="pt")

        return {
            "input_ids": txt["input_ids"].squeeze(0),
            "attention_mask": txt["attention_mask"].squeeze(0),
            "pixel_values": img["pixel_values"].squeeze(0),
            "labels": torch.tensor(label2id[int(row["prdtypecode"])])
        }

train_loader = DataLoader(RakutenDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(RakutenDataset(val_df), batch_size=BATCH_SIZE)

# =====================
# MODEL (GATED)
# =====================
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.text = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision = CLIPVisionModel.from_pretrained(CLIP_ID)
        self.debug_done = False

        self.gate = nn.Sequential(
            nn.Linear(1536, 512),
            nn.ReLU(),
            nn.Linear(512, 768),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(2304, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, i, a, p):
        t = self.text(input_ids=i, attention_mask=a).pooler_output
        v = self.vision(pixel_values=p).pooler_output

        t = t / t.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-12)

        fusion = torch.cat([t, v], dim=1)
        g = self.gate(fusion)

        fused = g * v + (1 - g) * t
        final = torch.cat([fused, t, v], dim=1)

        if not self.debug_done:
            print("t shape:", t.shape)
            print("v shape:", v.shape)
            print("fusion shape:", fusion.shape)
            print("g shape:", g.shape)
            print("final shape:", final.shape)
            self.debug_done = True

        return self.classifier(final)

model = FusionModel(num_classes=len(label2id)).to(DEVICE)

# =====================
# OPTIMIZER
# =====================
optimizer = torch.optim.AdamW([
    {"params": [p for n,p in model.named_parameters() if "classifier" not in n], "lr": BACKBONE_LR},
    {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR}
])

criterion = nn.CrossEntropyLoss()

# =====================
# TRAIN FUNCTION
# =====================
def run_epoch(loader, train=True):
    model.train() if train else model.eval()

    all_preds, all_true, all_logits = [], [], []
    total_loss = 0

    for step, batch in enumerate(tqdm(loader)):
        i = batch["input_ids"].to(DEVICE)
        a = batch["attention_mask"].to(DEVICE)
        p = batch["pixel_values"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(i, a, p)
            loss = criterion(logits, y)

            if train:
                (loss / ACCUM_STEPS).backward()
                if step % ACCUM_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad()

        total_loss += loss.item() * y.size(0)

        preds = torch.argmax(logits, 1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.cpu().numpy())

        if not train:
            all_logits.append(logits.detach().cpu().numpy())

    loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_true, all_preds)
    f1 = f1_score(all_true, all_preds, average="macro")

    logits_out = np.concatenate(all_logits) if not train else None

    return loss, acc, f1, logits_out, all_true, all_preds

# =====================
# TRAIN LOOP
# =====================
best_f1 = 0
early = 0

for epoch in range(1, MAX_EPOCHS+1):

    t_loss, t_acc, t_f1, _, _, _ = run_epoch(train_loader, True)
    v_loss, v_acc, v_f1, v_logits, y_true, y_pred = run_epoch(val_loader, False)

    print(f"Epoch {epoch} | Val F1: {v_f1:.4f}")

    if v_f1 > best_f1:
        best_f1 = v_f1
        early = 0

        torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pt")
        np.save(OUTPUT_DIR / "multimodal_logits.npy", v_logits)

    else:
        early += 1

    if early >= EARLY_STOPPING_PATIENCE:
        print("Early stopping")
        break


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

epochs = np.array([1,2,3,4,5,6])
val_f1 = np.array([0.8200,0.8502,0.8595,0.8640,0.8611,0.8601])

best_idx = np.argmax(val_f1)

plt.figure(figsize=(8,5))
plt.plot(epochs, val_f1, marker='o', label="Val F1")

plt.scatter(epochs[best_idx], val_f1[best_idx], s=100)
plt.text(epochs[best_idx], val_f1[best_idx]+0.002,
         f"Best: {val_f1[best_idx]:.4f} (epoch {epochs[best_idx]})",
         ha='center')

plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Model Performance (CamemBERT + CLIP Fusion)")
plt.legend()
plt.grid(False)
plt.tight_layout()

plt.show()

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import CamembertModel, AutoTokenizer, CLIPVisionModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

# =====================
# CONFIG
# =====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

BATCH_SIZE = 16
MAX_TEXT_LEN = 128

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"

# =====================
# PATHS
# =====================
BASE_DIR = Path.cwd().parent.parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

INTERPRET_DIR = BASE_DIR / "outputs" / "gradCam"
INTERPRET_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
INTERPRET_CSV_PATH = INTERPRET_DIR / "interpretation_results.csv"
TOP_TEXT_CSV_PATH = INTERPRET_DIR / "top_text_weighted_samples.csv"
TOP_VISION_CSV_PATH = INTERPRET_DIR / "top_vision_weighted_samples.csv"
SUMMARY_JSON_PATH = INTERPRET_DIR / "interpretation_summary.json"

# =====================
# LOAD DATA
# =====================
val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json", "r") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

id2label = {v: k for k, v in label2id.items()}

# =====================
# TOKENIZER / PROCESSOR
# =====================
tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
processor = CLIPProcessor.from_pretrained(CLIP_ID)

# =====================
# DATASET
# =====================
class RakutenDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = IMAGE_DIR / f"image_{row['imageid']}_product_{row['productid']}.jpg"
        image = Image.open(img_path).convert("RGB")

        designation = "" if pd.isna(row.get("designation", "")) else str(row.get("designation", ""))
        description = "" if pd.isna(row.get("description", "")) else str(row.get("description", ""))
        text = f"{designation} {description}".strip()

        txt = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_TEXT_LEN,
            return_tensors="pt"
        )

        img = processor(images=image, return_tensors="pt")

        return {
            "input_ids": txt["input_ids"].squeeze(0),
            "attention_mask": txt["attention_mask"].squeeze(0),
            "pixel_values": img["pixel_values"].squeeze(0),
            "labels": torch.tensor(label2id[int(row["prdtypecode"])], dtype=torch.long),
            "imageid": row["imageid"],
            "productid": row["productid"],
            "prdtypecode_raw": int(row["prdtypecode"]),
            "designation": designation,
            "description": description,
            "raw_text": text
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "imageid": [x["imageid"] for x in batch],
        "productid": [x["productid"] for x in batch],
        "prdtypecode_raw": [x["prdtypecode_raw"] for x in batch],
        "designation": [x["designation"] for x in batch],
        "description": [x["description"] for x in batch],
        "raw_text": [x["raw_text"] for x in batch]
    }

val_loader = DataLoader(
    RakutenDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# =====================
# MODEL
# =====================
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.text = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision = CLIPVisionModel.from_pretrained(CLIP_ID)

        self.gate = nn.Sequential(
            nn.Linear(1536, 512),
            nn.ReLU(),
            nn.Linear(512, 768),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(2304, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, i, a, p, return_interpret=False):
        text_outputs = self.text(input_ids=i, attention_mask=a)
        vision_outputs = self.vision(pixel_values=p)

        if text_outputs.pooler_output is not None:
            t = text_outputs.pooler_output
        else:
            t = text_outputs.last_hidden_state[:, 0, :]

        v = vision_outputs.pooler_output

        t = t / t.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-12)

        fusion = torch.cat([t, v], dim=1)
        g = self.gate(fusion)

        fused = g * v + (1.0 - g) * t
        final = torch.cat([fused, t, v], dim=1)
        logits = self.classifier(final)

        if return_interpret:
            return logits, t, v, g, fused

        return logits

# =====================
# LOAD BEST MODEL
# =====================
model = FusionModel(num_classes=len(label2id)).to(DEVICE)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model not found: {BEST_MODEL_PATH}")

state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

print("Loaded best model from:", BEST_MODEL_PATH)

# =====================
# INTERPRETATION
# =====================
all_rows = []
all_true = []
all_pred = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Interpreting"):
        i = batch["input_ids"].to(DEVICE)
        a = batch["attention_mask"].to(DEVICE)
        p = batch["pixel_values"].to(DEVICE)
        y = batch["labels"].to(DEVICE)

        logits, t, v, g, fused = model(i, a, p, return_interpret=True)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_true.extend(y.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())

        gate_mean = g.mean(dim=1).cpu().numpy()
        text_contribution = (1.0 - g).mean(dim=1).cpu().numpy()
        vision_contribution = g.mean(dim=1).cpu().numpy()
        confidence = probs.max(dim=1).values.cpu().numpy()

        for idx_in_batch in range(len(batch["imageid"])):
            true_label_id = int(y[idx_in_batch].cpu().item())
            pred_label_id = int(preds[idx_in_batch].cpu().item())

            row = {
                "imageid": batch["imageid"][idx_in_batch],
                "productid": batch["productid"][idx_in_batch],
                "true_label_id": true_label_id,
                "pred_label_id": pred_label_id,
                "true_prdtypecode": int(id2label[true_label_id]),
                "pred_prdtypecode": int(id2label[pred_label_id]),
                "correct": int(true_label_id == pred_label_id),
                "confidence": float(confidence[idx_in_batch]),
                "gate_mean": float(gate_mean[idx_in_batch]),
                "text_contribution": float(text_contribution[idx_in_batch]),
                "vision_contribution": float(vision_contribution[idx_in_batch]),
                "designation": batch["designation"][idx_in_batch],
                "description": batch["description"][idx_in_batch],
                "raw_text": batch["raw_text"][idx_in_batch]
            }

            all_rows.append(row)

# =====================
# SAVE RESULTS
# =====================
result_df = pd.DataFrame(all_rows)
result_df.to_csv(INTERPRET_CSV_PATH, index=False)

top_text_df = result_df.sort_values("text_contribution", ascending=False).head(50)
top_text_df.to_csv(TOP_TEXT_CSV_PATH, index=False)

top_vision_df = result_df.sort_values("vision_contribution", ascending=False).head(50)
top_vision_df.to_csv(TOP_VISION_CSV_PATH, index=False)

acc = accuracy_score(all_true, all_pred)
macro_f1 = f1_score(all_true, all_pred, average="macro")
report = classification_report(all_true, all_pred, output_dict=True, zero_division=0)

summary = {
    "num_samples": int(len(result_df)),
    "accuracy": float(acc),
    "macro_f1": float(macro_f1),
    "mean_gate": float(result_df["gate_mean"].mean()),
    "mean_text_contribution": float(result_df["text_contribution"].mean()),
    "mean_vision_contribution": float(result_df["vision_contribution"].mean()),
    "mean_gate_correct": float(result_df[result_df["correct"] == 1]["gate_mean"].mean()) if (result_df["correct"] == 1).any() else None,
    "mean_gate_wrong": float(result_df[result_df["correct"] == 0]["gate_mean"].mean()) if (result_df["correct"] == 0).any() else None
}

with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# =====================
# PRINT SUMMARY
# =====================
print("\nInterpretation finished.")
print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Mean gate value: {result_df['gate_mean'].mean():.4f}")
print(f"Mean text contribution: {result_df['text_contribution'].mean():.4f}")
print(f"Mean vision contribution: {result_df['vision_contribution'].mean():.4f}")

print("\nMost text-driven samples:")
print(top_text_df[[
    "imageid", "productid", "true_prdtypecode", "pred_prdtypecode",
    "confidence", "text_contribution", "vision_contribution", "correct"
]].head(10).to_string(index=False))

print("\nMost vision-driven samples:")
print(top_vision_df[[
    "imageid", "productid", "true_prdtypecode", "pred_prdtypecode",
    "confidence", "text_contribution", "vision_contribution", "correct"
]].head(10).to_string(index=False))

print("\nSaved files:")
print(INTERPRET_CSV_PATH)
print(TOP_TEXT_CSV_PATH)
print(TOP_VISION_CSV_PATH)
print(SUMMARY_JSON_PATH)

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import CamembertModel, AutoTokenizer, CLIPVisionModel, CLIPProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"
MAX_TEXT_LEN = 128

BASE_DIR = Path.cwd().parent.parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
OUTPUT_DIR = BASE_DIR / "outputs"
GRADCAM_DIR = OUTPUT_DIR / "gradCam"
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
VAL_SPLIT_PATH = SPLIT_DIR / "val_split.csv"
LABEL2ID_PATH = SPLIT_DIR / "label2id.json"

SAMPLE_INDEX = random.randint(0, len(val_df) - 1)


val_df = pd.read_csv(VAL_SPLIT_PATH)

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
processor = CLIPProcessor.from_pretrained(CLIP_ID)

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.text = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision = CLIPVisionModel.from_pretrained(CLIP_ID)
        self.gate = nn.Sequential(
            nn.Linear(1536, 512),
            nn.ReLU(),
            nn.Linear(512, 768),
            nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.Linear(2304, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values, return_interpret=False):
        text_outputs = self.text(input_ids=input_ids, attention_mask=attention_mask)
        vision_outputs = self.vision(pixel_values=pixel_values)

        t = text_outputs.pooler_output
        if t is None:
            t = text_outputs.last_hidden_state[:, 0, :]

        v = vision_outputs.pooler_output

        t = t / t.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-12)

        fusion = torch.cat([t, v], dim=1)
        g = self.gate(fusion)
        fused = g * v + (1.0 - g) * t
        final = torch.cat([fused, t, v], dim=1)
        logits = self.classifier(final)

        if return_interpret:
            return logits, g, t, v, fused

        return logits

class ViTGradCAM:
    def __init__(self, model, target_module):
        self.model = model
        self.target_module = target_module
        self.activations = None
        self.gradients = None
        self.forward_handle = target_module.register_forward_hook(self._forward_hook)
        self.backward_handle = target_module.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inputs, output):
        if isinstance(output, tuple):
            output = output[0]
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        grad = grad_output[0]
        if isinstance(grad, tuple):
            grad = grad[0]
        self.gradients = grad

    def remove(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(self, input_ids, attention_mask, pixel_values, target_class=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(input_ids, attention_mask, pixel_values)

        if target_class is None:
            target_class = logits.argmax(dim=1)

        score = logits.gather(1, target_class.view(-1, 1)).sum()
        score.backward()

        acts = self.activations
        grads = self.gradients

        if acts is None or grads is None:
            raise RuntimeError("Grad-CAM hooks did not capture activations/gradients.")

        acts = acts[:, 1:, :]
        grads = grads[:, 1:, :]

        weights = grads.mean(dim=1, keepdim=True)
        cam = torch.relu((acts * weights).sum(dim=-1))

        n_patches = cam.shape[1]
        side = int(n_patches ** 0.5)
        cam = cam.reshape(-1, 1, side, side)
        cam = F.interpolate(cam, size=pixel_values.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze(1)

        cam_min = cam.amin(dim=(1, 2), keepdim=True)
        cam_max = cam.amax(dim=(1, 2), keepdim=True)
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)

        return cam.detach().cpu().numpy(), logits.detach()

def build_image_path(row):
    return IMAGE_DIR / f"image_{row['imageid']}_product_{row['productid']}.jpg"

def load_sample(row):
    image_path = build_image_path(row)
    image = Image.open(image_path).convert("RGB")

    designation = "" if pd.isna(row.get("designation", "")) else str(row.get("designation", ""))
    description = "" if pd.isna(row.get("description", "")) else str(row.get("description", ""))
    text = f"{designation} {description}".strip()

    txt = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LEN,
        return_tensors="pt"
    )

    img = processor(images=image, return_tensors="pt")

    return image, text, txt, img, image_path

def overlay_heatmap_on_image(pil_image, cam_2d, alpha=0.45):
    image = np.array(pil_image).astype(np.float32) / 255.0

    cam_img = Image.fromarray((cam_2d * 255).astype(np.uint8))
    cam_img = cam_img.resize((pil_image.width, pil_image.height), resample=Image.BILINEAR)
    cam_resized = np.array(cam_img).astype(np.float32) / 255.0

    heatmap = plt.get_cmap("jet")(cam_resized)[..., :3]
    overlay = (1 - alpha) * image + alpha * heatmap
    overlay = np.clip(overlay, 0, 1)

    return overlay

model = FusionModel(num_classes=len(label2id)).to(DEVICE)
state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

target_layer = model.vision.vision_model.encoder.layers[-1]
gradcam = ViTGradCAM(model, target_layer)

row = val_df.iloc[SAMPLE_INDEX]
pil_image, raw_text, txt, img, image_path = load_sample(row)

input_ids = txt["input_ids"].to(DEVICE)
attention_mask = txt["attention_mask"].to(DEVICE)
pixel_values = img["pixel_values"].to(DEVICE)

with torch.enable_grad():
    logits, g, _, _, _ = model(input_ids, attention_mask, pixel_values, return_interpret=True)
    probs = torch.softmax(logits, dim=1)
    pred_id = int(torch.argmax(probs, dim=1).item())
    confidence = float(torch.max(probs).item())
    cam, _ = gradcam.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        target_class=torch.tensor([pred_id], device=DEVICE)
    )

gate_mean = float(g.mean().item())
vision_contribution = gate_mean
text_contribution = 1.0 - gate_mean

pred_prdtypecode = id2label[pred_id]
true_prdtypecode = int(row["prdtypecode"])

overlay = overlay_heatmap_on_image(pil_image, cam[0], alpha=0.45)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor="white")

axes[0].imshow(pil_image)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(overlay)
axes[1].set_title("Grad-CAM on Vision Branch")
axes[1].axis("off")

axes[2].bar(["Text", "Vision"], [text_contribution, vision_contribution])
axes[2].set_ylim(0, 1)
axes[2].set_title("Gate Contribution")
axes[2].set_ylabel("Weight")

plt.suptitle(
    f"True: {true_prdtypecode} | Pred: {pred_prdtypecode} | Confidence: {confidence:.4f}",
    fontsize=12
)
plt.tight_layout()

figure_path = GRADCAM_DIR / f"gradcam_gate_sample_{SAMPLE_INDEX}.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

result = {
    "sample_index": int(SAMPLE_INDEX),
    "image_path": str(image_path),
    "true_prdtypecode": true_prdtypecode,
    "pred_prdtypecode": int(pred_prdtypecode),
    "confidence": confidence,
    "text_contribution": text_contribution,
    "vision_contribution": vision_contribution,
    "raw_text": raw_text
}

result_path = GRADCAM_DIR / f"gradcam_gate_sample_{SAMPLE_INDEX}.json"
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("Saved figure:", figure_path)
print("Saved metadata:", result_path)
print("True label:", true_prdtypecode)
print("Pred label:", pred_prdtypecode)
print("Confidence:", round(confidence, 4))
print("Text contribution:", round(text_contribution, 4))
print("Vision contribution:", round(vision_contribution, 4))

gradcam.remove()

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import CamembertModel, AutoTokenizer, CLIPVisionModel, CLIPProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"
MAX_TEXT_LEN = 128

BASE_DIR = Path.cwd().parent.parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
OUTPUT_DIR = BASE_DIR / "outputs"
GRADCAM_DIR = OUTPUT_DIR / "gradCam"
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
VAL_SPLIT_PATH = SPLIT_DIR / "val_split.csv"
LABEL2ID_PATH = SPLIT_DIR / "label2id.json"

val_df = pd.read_csv(VAL_SPLIT_PATH)
SAMPLE_INDEX = random.randint(0, len(val_df) - 1)

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
processor = CLIPProcessor.from_pretrained(CLIP_ID)

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.text = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision = CLIPVisionModel.from_pretrained(CLIP_ID)
        self.gate = nn.Sequential(
            nn.Linear(1536, 512),
            nn.ReLU(),
            nn.Linear(512, 768),
            nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.Linear(2304, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values, return_interpret=False):
        text_outputs = self.text(input_ids=input_ids, attention_mask=attention_mask)
        vision_outputs = self.vision(pixel_values=pixel_values)

        t = text_outputs.pooler_output
        if t is None:
            t = text_outputs.last_hidden_state[:, 0, :]

        v = vision_outputs.pooler_output

        t = t / t.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-12)

        fusion = torch.cat([t, v], dim=1)
        g = self.gate(fusion)
        fused = g * v + (1.0 - g) * t
        final = torch.cat([fused, t, v], dim=1)
        logits = self.classifier(final)

        if return_interpret:
            return logits, g, t, v, fused

        return logits

class ViTGradCAM:
    def __init__(self, model, target_module):
        self.model = model
        self.target_module = target_module
        self.activations = None
        self.gradients = None
        self.forward_handle = target_module.register_forward_hook(self._forward_hook)
        self.backward_handle = target_module.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inputs, output):
        if isinstance(output, tuple):
            output = output[0]
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        grad = grad_output[0]
        if isinstance(grad, tuple):
            grad = grad[0]
        self.gradients = grad

    def remove(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(self, input_ids, attention_mask, pixel_values, target_class=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(input_ids, attention_mask, pixel_values)

        if target_class is None:
            target_class = logits.argmax(dim=1)

        score = logits.gather(1, target_class.view(-1, 1)).sum()
        score.backward()

        acts = self.activations
        grads = self.gradients

        if acts is None or grads is None:
            raise RuntimeError("Grad-CAM hooks did not capture activations/gradients.")

        if acts.dim() != 3 or grads.dim() != 3:
            raise RuntimeError(f"Expected 3D tensors, got acts={acts.shape}, grads={grads.shape}")

        acts = acts[:, 1:, :]
        grads = grads[:, 1:, :]

        weights = grads.mean(dim=1, keepdim=True)
        cam = torch.relu((acts * weights).sum(dim=-1))

        n_patches = cam.shape[1]
        side = int(n_patches ** 0.5)

        if side * side != n_patches:
            raise RuntimeError(f"Patch count is not square: {n_patches}")

        cam_patch = cam.reshape(-1, side, side)

        cam_up = cam_patch.unsqueeze(1)
        cam_up = F.interpolate(
            cam_up,
            size=pixel_values.shape[-2:],
            mode="bilinear",
            align_corners=False
        ).squeeze(1)

        cam_min = cam_up.amin(dim=(1, 2), keepdim=True)
        cam_max = cam_up.amax(dim=(1, 2), keepdim=True)
        cam_up = (cam_up - cam_min) / (cam_max - cam_min + 1e-8)

        cam_patch_min = cam_patch.amin(dim=(1, 2), keepdim=True)
        cam_patch_max = cam_patch.amax(dim=(1, 2), keepdim=True)
        cam_patch = (cam_patch - cam_patch_min) / (cam_patch_max - cam_patch_min + 1e-8)

        return cam_patch.detach().cpu().numpy(), cam_up.detach().cpu().numpy(), logits.detach()

def build_image_path(row):
    return IMAGE_DIR / f"image_{row['imageid']}_product_{row['productid']}.jpg"

def load_sample(row):
    image_path = build_image_path(row)
    image = Image.open(image_path).convert("RGB")

    designation = "" if pd.isna(row.get("designation", "")) else str(row.get("designation", ""))
    description = "" if pd.isna(row.get("description", "")) else str(row.get("description", ""))
    text = f"{designation} {description}".strip()

    txt = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LEN,
        return_tensors="pt"
    )

    img = processor(images=image, return_tensors="pt")

    return image, text, txt, img, image_path

def overlay_heatmap_on_image(pil_image, cam_2d, alpha=0.65):
    image = np.array(pil_image).astype(np.float32) / 255.0

    cam_img = Image.fromarray((cam_2d * 255).astype(np.uint8))
    cam_img = cam_img.resize((pil_image.width, pil_image.height), resample=Image.BILINEAR)
    cam_resized = np.array(cam_img).astype(np.float32) / 255.0

    heatmap = plt.get_cmap("jet")(cam_resized)[..., :3]
    overlay = (1.0 - alpha) * image + alpha * heatmap
    overlay = np.clip(overlay, 0, 1)

    return overlay

model = FusionModel(num_classes=len(label2id)).to(DEVICE)
state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

target_layer = model.vision.vision_model.encoder.layers[-1].layer_norm1
gradcam = ViTGradCAM(model, target_layer)

row = val_df.iloc[SAMPLE_INDEX]
pil_image, raw_text, txt, img, image_path = load_sample(row)

input_ids = txt["input_ids"].to(DEVICE)
attention_mask = txt["attention_mask"].to(DEVICE)
pixel_values = img["pixel_values"].to(DEVICE)

with torch.enable_grad():
    logits, g, _, _, _ = model(
        input_ids,
        attention_mask,
        pixel_values,
        return_interpret=True
    )
    probs = torch.softmax(logits, dim=1)
    pred_id = int(torch.argmax(probs, dim=1).item())
    confidence = float(torch.max(probs).item())

    cam_patch, cam_up, _ = gradcam.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        target_class=torch.tensor([pred_id], device=DEVICE)
    )

print("cam_patch min:", float(cam_patch[0].min()))
print("cam_patch max:", float(cam_patch[0].max()))
print("cam_patch mean:", float(cam_patch[0].mean()))
print("cam_up min:", float(cam_up[0].min()))
print("cam_up max:", float(cam_up[0].max()))
print("cam_up mean:", float(cam_up[0].mean()))

gate_mean = float(g.mean().item())
vision_contribution = gate_mean
text_contribution = 1.0 - gate_mean

pred_prdtypecode = id2label[pred_id]
true_prdtypecode = int(row["prdtypecode"])

overlay = overlay_heatmap_on_image(pil_image, cam_up[0], alpha=0.65)

fig, axes = plt.subplots(1, 4, figsize=(22, 5), facecolor="white")

axes[0].imshow(pil_image)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(cam_patch[0], cmap="jet")
axes[1].set_title("Raw Patch CAM")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title("Overlay Grad-CAM")
axes[2].axis("off")

axes[3].bar(["Text", "Vision"], [text_contribution, vision_contribution])
axes[3].set_ylim(0, 1)
axes[3].set_title("Gate Contribution")
axes[3].set_ylabel("Weight")

plt.suptitle(
    f"True: {true_prdtypecode} | Pred: {pred_prdtypecode} | Confidence: {confidence:.4f}",
    fontsize=12
)
plt.tight_layout()
plt.show()

figure_path = GRADCAM_DIR / f"gradcam_debug_sample_{SAMPLE_INDEX}.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")

result = {
    "sample_index": int(SAMPLE_INDEX),
    "image_path": str(image_path),
    "true_prdtypecode": true_prdtypecode,
    "pred_prdtypecode": int(pred_prdtypecode),
    "confidence": confidence,
    "text_contribution": text_contribution,
    "vision_contribution": vision_contribution,
    "raw_text": raw_text,
    "cam_patch_min": float(cam_patch[0].min()),
    "cam_patch_max": float(cam_patch[0].max()),
    "cam_patch_mean": float(cam_patch[0].mean()),
    "cam_up_min": float(cam_up[0].min()),
    "cam_up_max": float(cam_up[0].max()),
    "cam_up_mean": float(cam_up[0].mean())
}

result_path = GRADCAM_DIR / f"gradcam_debug_sample_{SAMPLE_INDEX}.json"
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("Saved figure:", figure_path)
print("Saved metadata:", result_path)
print("True label:", true_prdtypecode)
print("Pred label:", pred_prdtypecode)
print("Confidence:", round(confidence, 4))
print("Text contribution:", round(text_contribution, 4))
print("Vision contribution:", round(vision_contribution, 4))

gradcam.remove()

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import CamembertModel, AutoTokenizer, CLIPVisionModel, CLIPProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", DEVICE)

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"
MAX_TEXT_LEN = 128
NUM_WRONG_SAMPLES = 5

BASE_DIR = Path.cwd().parent.parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
OUTPUT_DIR = BASE_DIR / "outputs"
WRONG_CAM_DIR = OUTPUT_DIR / "gradCam_wrong_predictions"
WRONG_CAM_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pt"
VAL_SPLIT_PATH = SPLIT_DIR / "val_split.csv"
LABEL2ID_PATH = SPLIT_DIR / "label2id.json"

val_df = pd.read_csv(VAL_SPLIT_PATH)

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    loaded = json.load(f)
    label2id = {int(k): int(v) for k, v in loaded.items()}

id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
processor = CLIPProcessor.from_pretrained(CLIP_ID)

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.text = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision = CLIPVisionModel.from_pretrained(CLIP_ID)
        self.gate = nn.Sequential(
            nn.Linear(1536, 512),
            nn.ReLU(),
            nn.Linear(512, 768),
            nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.Linear(2304, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values, return_interpret=False):
        text_outputs = self.text(input_ids=input_ids, attention_mask=attention_mask)
        vision_outputs = self.vision(pixel_values=pixel_values)

        t = text_outputs.pooler_output
        if t is None:
            t = text_outputs.last_hidden_state[:, 0, :]
